# Week 12 — Neural Network Surrogate Training (FINAL ROUND)

Same architecture family as W4-W11: MLP with H in {16, 32}, four regularisation variants, 5-fold CV across 8 configs.

Re-trains on the latest data including W11 portal returns (21/21/26/41/31/31/41/51 pts).

W11 produced 5 new bests (F2, F3, F4, F7, F8); F1 sign confirmed positive at the Gaussian center.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_12')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y


In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta


## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()


F1: 21 pts, 2D, baseline RMSE = 0.0016
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.0025 ← BEST
    H= 32  dropout     CV RMSE = 0.0027
    H= 32  ensemble    CV RMSE = 0.0035
    H= 16  ensemble    CV RMSE = 0.0037
    H= 32  plain       CV RMSE = 0.0038
    H= 32  wd          CV RMSE = 0.0039
    H= 16  wd          CV RMSE = 0.0042
    H= 16  plain       CV RMSE = 0.0043
  → best: H=16, dropout, RMSE=0.0025 (-57.9% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.014000000432133675, -0.008999999612569809]



F2: 21 pts, 2D, baseline RMSE = 0.2437
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.1497 ← BEST
    H= 16  dropout     CV RMSE = 0.1719
    H= 32  plain       CV RMSE = 0.2383
    H= 32  wd          CV RMSE = 0.2409
    H= 16  ensemble    CV RMSE = 0.2606
    H= 16  wd          CV RMSE = 0.2608
    H= 16  plain       CV RMSE = 0.2640
    H= 32  ensemble    CV RMSE = 0.2739
  → best: H=32, dropout, RMSE=0.1497 (+38.6% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.1459999978542328, 0.5860000252723694]



F3: 26 pts, 3D, baseline RMSE = 0.0728
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0772 ← BEST
    H= 16  ensemble    CV RMSE = 0.0778
    H= 16  plain       CV RMSE = 0.0862
    H= 32  wd          CV RMSE = 0.0873
    H= 32  dropout     CV RMSE = 0.0879
    H= 16  dropout     CV RMSE = 0.0893
    H= 32  plain       CV RMSE = 0.0905
    H= 16  wd          CV RMSE = 0.1029
  → best: H=32, ensemble, RMSE=0.0772 (-6.1% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.04500000178813934, -0.09000000357627869, -0.026000000536441803]



F4: 41 pts, 4D, baseline RMSE = 9.8031
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 4.4397 ← BEST
    H= 32  wd          CV RMSE = 4.9188
    H= 32  plain       CV RMSE = 4.9265
    H= 16  plain       CV RMSE = 4.9718
    H= 16  wd          CV RMSE = 5.0134
    H= 32  ensemble    CV RMSE = 5.1720
    H= 32  dropout     CV RMSE = 5.4649
    H= 16  dropout     CV RMSE = 6.0200
  → best: H=16, ensemble, RMSE=4.4397 (+54.7% vs baseline) ✓ beats baseline


...

  Gradient dY/dx at best point: [-0.2549999952316284, -0.6269999742507935, 0.06700000166893005, -0.10000000149011612, -0.9300000071525574]



F7: 41 pts, 6D, baseline RMSE = 0.7173
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.3421 ← BEST
    H= 32  dropout     CV RMSE = 0.3709
    H= 16  plain       CV RMSE = 0.3978
    H= 16  wd          CV RMSE = 0.4044
    H= 16  ensemble    CV RMSE = 0.4122
    H= 16  dropout     CV RMSE = 0.4373
    H= 32  wd          CV RMSE = 0.4637
    H= 32  plain       CV RMSE = 0.4751
  → best: H=32, ensemble, RMSE=0.3421 (+52.3% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.8080000281333923, -3.3949999809265137, 2.813999891281128, 0.6539999842643738, -5.296000003814697, -0.1289999932050705]



F8: 51 pts, 8D, baseline RMSE = 1.1900
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.4379 ← BEST
    H= 32  wd          CV RMSE = 0.4550
    H= 16  dropout     CV RMSE = 0.4558
    H= 32  plain       CV RMSE = 0.4604
    H= 32  dropout     CV RMSE = 0.4655
    H= 16  ensemble    CV RMSE = 0.4762
    H= 16  plain       CV RMSE = 0.5413
    H= 16  wd          CV RMSE = 0.5475
  → best: H=32, ensemble, RMSE=0.4379 (+63.2% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [0.04699999839067459, 0.1599999964237213, -0.3869999945163727, 0.04399999976158142, 0.25, -0.039000000804662704, -0.3109999895095825, -0.2029999941587448]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — now especially useful since W10 confirmed F1 is deterministic (W3's 3.65e-7 is a real reproducible peak, not noise).


In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')


F1: 21 pts, 15 positive, 6 zero-or-negative (71% positive)


Sign classifier trained. LOO accuracy = 71.43%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')


 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  dropout         0.0025      0.0016    -57.9%       ✗
 2   32  dropout         0.1497      0.2437    +38.6%       ✓
 3   32  ensemble        0.0772      0.0728     -6.1%       ✗
 4   16  ensemble        4.4397      9.8031    +54.7%       ✓
 5   16  ensemble      278.0937   1987.9385    +86.0%       ✓
 6   32  ensemble        0.2883      0.6723    +57.1%       ✓
 7   32  ensemble        0.3421      0.7173    +52.3%       ✓
 8   32  ensemble        0.4379      1.1900    +63.2%       ✓
